In [1]:
import winsound
from tqdm.auto import tqdm
import sympy as sp
from sympy.core.function import AppliedUndef
from sympy.physics.wigner import clebsch_gordan
from sympy.physics.quantum.spin import Rotation as rot
from sympy import pi
from random import randint
import psutil
import dill
import os

sp.init_printing(use_unicode=True, wrap_line=True, num_columns=120)

Jmax = 2
Lmax = 2

In [2]:
# --- Definizione simboli ---
J, LB, LR, L1, L2, JX, lam, lam1, lam2, eta = sp.symbols('J LB LR L1 L2 JX lam lam1 lam2 eta', integer=True)
lambda1, lambda2 = sp.symbols('lambda1 lambda2', real=True)
theta1, theta2, thetal, phi, chi = sp.symbols('theta1 theta2 thetal phi chi', real=True)
eta_par = sp.symbols('eta_par', integer=True)
Res = sp.symbols('Res')

# --- Helicity amplitudes e altri coefficienti simbolici ---
H = sp.Function('H', complex=True)(eta, J, lam, LB, Res)     # Helicity amplitude per B -> K_J
Hbar = sp.Function('Hbar', complex=True)(eta, J, lam, LB, Res)     # Helicity amplitude per B -> K*_J
LHA = sp.Function('LHA', real=True)(eta_par, lam1, lam2)  # Leptonic Helicity amplitude
G = sp.Function('G', complex=True)(eta, J, lam, LB, Res)  # Helicity Amplitude for K_J -> R P1

# --- Breit-Wigner propagatore ---
Prop = sp.Function('R')(Res)

# --- Blatt-Weisskopf factor ---
L, rBW = sp.symbols('L rBW', positive=True, real=True)
rB, r3, r2 = sp.symbols('rB r3 r2', positive=True, real=True)
q = sp.symbols('q', real=True)
qR = sp.symbols('qR', real=True)

# funzione simbolica generica
BWfunc = sp.Function('BW', real=True)

# istanza base (con argomenti simbolici)
BW = BWfunc(L, rBW, qR)

atot = []
a_NR = sp.symbols('a_NR', complex=True)
atot.append(a_NR)  # contributo non-risonante

In [3]:
### Cella per costruire l'ampiezza A3 ###

# Lista delle risonanze come tuple (nome, spin)
#(R, J, LR, JX, a_i), dove LR è il mom. ang. orb. tra P1 e (P2P3), JX è il mom. ang. orb. tra P2 e P3 (uguale allo spin JX).
list_reso3 = ['K(1460)', 'K1(1650)', 'K2(1770)', 'K2(1820)']
resonances3 = {
    ('K(1460)', 0, 0, 0): sp.symbols(f'a1', complex=True),
    ('K1(1650)', 1, 0, 1): sp.symbols(f'a2', complex=True),
    ('K1(1650)', 1, 1, 0): sp.symbols(f'a3', complex=True),
    ('K2(1820)', 2, 0, 2): sp.symbols(f'a4', complex=True),
    ('K2(1820)', 2, 2, 0): sp.symbols(f'a5', complex=True),
}

a3 = []
for a in resonances3.values():
    a3.append(a)
atot.extend(a3)

# --- Costruzione dell'ampiezza A3 ---
A3 = 0  # Inizializza l'ampiezza

for J_val in range(0, Jmax + 1):
    for LB_val in range(abs(J_val-1), abs(J_val-1) + 1):
        print(f"\n\nCalcolando contributi per J = {J_val}, LB = {LB_val}")
        # Somma su tutte le risonanze R con spin = J_val
        for idx_res, keys in enumerate(resonances3.items()):
            R_name, spin, LR, JX = keys[0]
            coeff_a = keys[1]
            if spin == J_val:
                R_symbol = sp.Symbol(R_name)
                for JX_val in [JX]:
                    for lamx_val in range(-JX_val, JX_val + 1):
                        for LR_val in range(abs(J_val - JX_val), abs(J_val - JX_val) + 1):
                            for lam_val in range(-min(1,J_val), min(1,J_val) + 1):
                                H_term = H.subs(J, J_val).subs(LB, LB_val).subs(lam, lam_val).subs(Res, R_symbol)
                                # usa il coefficiente corrispondente dalla lista a3
                                term = (
                                    coeff_a * H_term * clebsch_gordan(J_val, 1, LB_val, lam_val, -lam_val, 0) *
                                    LHA.subs(eta_par, eta).subs(lam1, lambda1).subs(lam2, lambda2) *
                                    rot.d(1, lam_val, lambda1-lambda2, thetal) * 
                                    sp.sqrt(sp.Rational((2*LR_val + 1),(2*J_val + 1))) * 
                                    sp.sqrt(sp.Rational((2*J_val + 1),4)/pi) * sp.sqrt(sp.Rational((2*JX_val + 1),4)/pi) * 
                                    clebsch_gordan(LR_val, JX_val, J_val, lam_val - lamx_val, lamx_val, lam_val) *
                                    rot.d(J_val, lam_val, lamx_val, theta1) * rot.d(JX_val, lamx_val, 0, theta2) * 
                                    (sp.cos(lam_val*phi) + sp.I*sp.sin(lam_val*phi)) * (sp.cos(lamx_val*chi) + sp.I*sp.sin(lamx_val*chi)) *
                                    Prop.subs(Res, R_symbol) *
                                    BW.subs(L, LB_val).subs(rBW, rB).subs(qR, sp.Symbol('B')) * BW.subs(L, J_val).subs(rBW, r3).subs(qR, R_symbol)
                                    )
                                A3 += term
                                print(f"\nR = {R_name}, (spin = {spin})")
                                print(f"Processando {R_name} (spin {J_val}) lambda = {lam_val}, LB = {LB_val}")
                                print(f"JX = {JX_val}, lambdaX = {lamx_val}, LR = {LR_val}")
                                print(f"term = {term}")

# Mostra l'ampiezza totale
print("\n" + "="*50)
print("Ampiezza A3 totale:")
if A3 == 0:
    print("L'ampiezza A3 è nulla (nessuna risonanza selezionata).")
else:
    print(f"Termini nell'ampiezza A3: {len(A3.as_ordered_terms())}")
print("="*50)

# Mostra l'ampiezza
sp.pprint(A3)



Calcolando contributi per J = 0, LB = 1

R = K(1460), (spin = 0)
Processando K(1460) (spin 0) lambda = 0, LB = 1
JX = 0, lambdaX = 0, LR = 0
term = a1*BW(0, r3, K(1460))*BW(1, rB, B)*H(eta, 0, 0, 1, K(1460))*LHA(eta, lambda1, lambda2)*R(K(1460))*WignerD(0, 0, 0, 0, theta1, 0)*WignerD(0, 0, 0, 0, theta2, 0)*WignerD(1, 0, lambda1 - lambda2, 0, thetal, 0)/(4*pi)


Calcolando contributi per J = 1, LB = 0

R = K1(1650), (spin = 1)
Processando K1(1650) (spin 1) lambda = -1, LB = 0
JX = 1, lambdaX = -1, LR = 0
term = a2*(-I*sin(chi) + cos(chi))*(-I*sin(phi) + cos(phi))*BW(0, rB, B)*BW(1, r3, K1(1650))*H(eta, 1, -1, 0, K1(1650))*LHA(eta, lambda1, lambda2)*R(K1(1650))*WignerD(1, -1, -1, 0, theta1, 0)*WignerD(1, -1, 0, 0, theta2, 0)*WignerD(1, -1, lambda1 - lambda2, 0, thetal, 0)/(4*pi)

R = K1(1650), (spin = 1)
Processando K1(1650) (spin 1) lambda = 0, LB = 0
JX = 1, lambdaX = -1, LR = 0
term = 0

R = K1(1650), (spin = 1)
Processando K1(1650) (spin 1) lambda = 1, LB = 0
JX = 1, lambdaX = -1, 

In [5]:
### Cella per costruire l'ampiezza AR ###

# Lista delle risonanze come tuple (nome, spin)
list_resoR = ['rho(770)', 'omega(782)', 'phi(1020)', 'f2(1270)', 'f0(1500)', 'f2(1525)',
              'K0*(700)', 'K*(892)', 'K*(1410)', 'K0*(1430)', 'K2*(1430)', 'K*(1680)', 'K0*(1950)']
resonancesR = {
    #(R, spin, L1, L2), dove L1 è il mom. ang. orb. tra P1 e R, R2 è uguale allo spin di R.
    ('rho(770)', 1, 0, 1): sp.symbols(f'a6', complex=True), #8
    ('omega(782)', 1, 0, 1): sp.symbols(f'a7', complex=True), #9
    ('phi(1020)', 1, 0, 1): sp.symbols(f'a8', complex=True), #10
    ('f2(1270)', 2, 0, 2): sp.symbols(f'a9', complex=True), #11
    ('f0(1500)', 0, 0, 0): sp.symbols(f'a10', complex=True), #12
    ('K*(892)', 1, 0, 1): sp.symbols(f'a11', complex=True), #15
    ('K*(1410)', 1, 0, 1): sp.symbols(f'a12', complex=True), #16
    ('K0*(1430)', 0, 0, 0): sp.symbols(f'a13', complex=True), #17
    ('K2*(1430)', 2, 0, 2): sp.symbols(f'a14', complex=True), #18
    ('K*(1680)', 1, 0, 1): sp.symbols(f'a15', complex=True), #19
    ('K0*(1950)', 0, 0, 0): sp.symbols(f'a16', complex=True), #20
}

aR = []
for a in resonancesR.values():
    aR.append(a)
atot.extend(aR)

# --- Costruzione dell'ampiezza AR ---
AR = 0
for J_val in range(0, Jmax + 1):
    for LB_val in range(abs(J_val-1), abs(J_val-1) + 1):
        print(f"\n\nCalcolando contributi per J = {J_val}, LB = {LB_val}")
        for idx_res, keys in enumerate(resonancesR.items()):
            R_name, spin, L1, L2 = keys[0]
            coeff_a = keys[1]
            if spin == J_val:
                R_symbol = sp.Symbol(R_name)    
                for lam_val in range(-min(1,J_val), min(1,J_val) + 1):
                    G_term = G.subs(J, J_val).subs(LB, LB_val).subs(lam, lam_val).subs(Res, R_symbol)
                    coeff_a = aR[idx_res]
                    term = (
                        coeff_a * G_term * clebsch_gordan(J_val, 1, LB_val, lam_val, -lam_val, 0) *
                        LHA.subs(eta_par, eta).subs(lam1, lambda1).subs(lam2, lambda2) *
                        rot.d(1, lam_val, lambda1-lambda2, thetal) * 
                        sp.sqrt(sp.Rational(1,4)/pi) * sp.sqrt(sp.Rational((2*J_val + 1),4)/pi) * 
                        rot.d(J_val, lam_val, lam_val, theta1) * rot.d(J_val, lam_val, 0, theta2) * 
                        (sp.cos(lam_val*phi) + sp.I*sp.sin(lam_val*phi)) * (sp.cos(lam_val*chi) + sp.I*sp.sin(lam_val*chi)) *
                        Prop.subs(Res, R_symbol) *
                        BW.subs(L, LB_val).subs(rBW, rB).subs(qR, sp.Symbol('B')) * BW.subs(L, J_val).subs(rBW, r3).subs(qR, R_symbol)
                        )
                    AR += term
                    print(f"\nR = {R_name}, (spin = {spin})")
                    print(f"Processando {R_name} (spin {J_val}) lambda = {lam_val}, LB = {LB_val}")
                    print(f"term = {term}")


# Mostra l'ampiezza totale
print("\n" + "="*50)
if AR == 0:
    print("L'ampiezza AR è nulla (nessuna risonanza selezionata).")
else:
    print("Ampiezza AR totale:")
    print(f"Termini nell'ampiezza AR: {len(AR.as_ordered_terms())}")
print("="*50)

# Mostra l'ampiezza
sp.pprint(AR)



Calcolando contributi per J = 0, LB = 1

R = f0(1500), (spin = 0)
Processando f0(1500) (spin 0) lambda = 0, LB = 1
term = a10*BW(0, r3, f0(1500))*BW(1, rB, B)*G(eta, 0, 0, 1, f0(1500))*LHA(eta, lambda1, lambda2)*R(f0(1500))*WignerD(0, 0, 0, 0, theta1, 0)*WignerD(0, 0, 0, 0, theta2, 0)*WignerD(1, 0, lambda1 - lambda2, 0, thetal, 0)/(4*pi)

R = K0*(1430), (spin = 0)
Processando K0*(1430) (spin 0) lambda = 0, LB = 1
term = a13*BW(0, r3, K0*(1430))*BW(1, rB, B)*G(eta, 0, 0, 1, K0*(1430))*LHA(eta, lambda1, lambda2)*R(K0*(1430))*WignerD(0, 0, 0, 0, theta1, 0)*WignerD(0, 0, 0, 0, theta2, 0)*WignerD(1, 0, lambda1 - lambda2, 0, thetal, 0)/(4*pi)

R = K0*(1950), (spin = 0)
Processando K0*(1950) (spin 0) lambda = 0, LB = 1
term = a16*BW(0, r3, K0*(1950))*BW(1, rB, B)*G(eta, 0, 0, 1, K0*(1950))*LHA(eta, lambda1, lambda2)*R(K0*(1950))*WignerD(0, 0, 0, 0, theta1, 0)*WignerD(0, 0, 0, 0, theta2, 0)*WignerD(1, 0, lambda1 - lambda2, 0, thetal, 0)/(4*pi)


Calcolando contributi per J = 1, LB = 0

R = r

In [6]:
### Cella per costruire l'ampiezza AR1R2 ###
list_resoR1R2 = ['K(1460)', 'K1(1270)', 'K1(1400)', 'K*(1410)', 'K1(1650)', 'K*(1680)', 
              'K2*(1430)', 'K2(1700)', 'K2(1820)', 'K2*(1980)']
# Dizionario delle risonanze che servono per identificare il canale
#(R1, spin1, R2, spin2, tipoR1, L1, L2): coefficient
resonancesR1R2 = {
    ('K(1460)', 0, 'rho(770)', 1, 'K', 1, 1): sp.symbols(f'a17', complex=True), 
    ('K(1460)', 0, 'K*(892)', 1, 'K', 1, 1): sp.symbols(f'a18', complex=True), 
    ('K1(1270)', 1, 'rho(770)', 1, 'K', 0, 1): sp.symbols(f'a19', complex=True), 
    ('K1(1270)', 1, 'omega(782)', 1, 'K', 0, 1): sp.symbols(f'a20', complex=True),
    ('K1(1270)', 1, 'K*(892)', 1, 'K', 0, 1): sp.symbols(f'a21', complex=True), 
    ('K1(1400)', 1, 'K*(892)', 1, 'K', 0, 1): sp.symbols(f'a22', complex=True), 
    ('K*(1410)', 1, 'K*(892)', 1, 'K*', 1, 1): sp.symbols(f'a23', complex=True), 
    ('K*(1680)', 1, 'rho(770)', 1, 'K*', 1, 1): sp.symbols(f'a24', complex=True),
    ('K*(1680)', 1, 'K*(892)', 1, 'K*', 1, 1): sp.symbols(f'a25', complex=True), 
    ('K2*(1430)', 2, 'rho(770)', 1, 'K*', 2, 1): sp.symbols(f'a26', complex=True), 
    ('K2*(1430)', 2, 'omega(782)', 1, 'K*', 2, 1): sp.symbols(f'a27', complex=True),
    ('K2*(1430)', 2, 'K*(892)', 1, 'K*', 2, 1): sp.symbols(f'a28', complex=True),
    ('K2(1700)', 2, 'K2*(1430)', 2, 'K', 0, 2): sp.symbols(f'a29', complex=True),
    ('K2(1700)', 2, 'K*(892)', 1, 'K', 1, 1): sp.symbols(f'a30', complex=True),
    ('K2(1820)', 2, 'K2*(1430)', 2, 'K', 0, 2): sp.symbols(f'a31', complex=True),
    ('K2(1820)', 2, 'phi(1020)', 1, 'K', 1, 1): sp.symbols(f'a32', complex=True),
    ('K2*(1980)', 2, 'phi(1020)', 1, 'K*', 2, 1): sp.symbols(f'a33', complex=True), 
}

# Dizionario dei coefficienti a(R1,R2) per ogni coppia di risonanze (complessi)
aR1R2 = []
for a in resonancesR1R2.values():
    aR1R2.append(a)
atot.extend(aR1R2)

# --- Costruzione dell'ampiezza AR1R2 ---
AR1R2 = 0
for J1_val in range (0, Jmax + 1):
    for LB_val in range(abs(J1_val-1), abs(J1_val-1) + 1):
        print(f"\n\nCalcolando contributi per J1 = {J1_val}, LB = {LB_val}")
        for key, coeff_a in resonancesR1R2.items():
            R1_name, R1_spin, R2_name, R2_spin, R1_type, L1, L2 = key
            if R1_spin == J1_val:  # Solo se lo spin corrisponde a J1_val
                if R1_type == 'K': auxH = H
                if R1_type == 'K*': auxH = Hbar
                R1_symbol = sp.Symbol(R1_name)
                R2_symbol = sp.Symbol(R2_name)
                J2_val = L2  # Considera solo J2_val = L2
                if R2_spin == J2_val:
                    for L1_val in [L1]:
                        for lam_val in range(-min(1,J1_val), min(1,J1_val) + 1):
                            Hterm = auxH.subs(J, J1_val).subs(LB, LB_val).subs(lam, lam_val).subs(Res, R1_symbol)
                            for lam2_val in range(-min(1,J2_val), min(1,J2_val) + 1):
                                term = (
                                    coeff_a * Hterm * clebsch_gordan(J1_val, 1, LB_val, lam_val, -lam_val, 0) *
                                        LHA.subs(eta_par, eta).subs(lam1, lambda1).subs(lam2, lambda2) *
                                        rot.d(1, lam_val, lambda1-lambda2, thetal) * 
                                        sp.sqrt(sp.Rational((2*J1_val + 1),4)/pi) * sp.sqrt(sp.Rational((2*J2_val + 1),4)/pi) * 
                                        sp.sqrt(sp.Rational((2*L1_val + 1),(2*J1_val + 1))) * 
                                        clebsch_gordan(L1_val, J2_val, J1_val, lam_val-lam2_val, lam2_val, lam_val) *
                                        rot.d(J1_val, lam_val, lam2_val, theta1) * rot.d(J2_val, lam2_val, 0, theta2) * 
                                        (sp.cos(lam_val*phi) + sp.I*sp.sin(lam_val*phi)) *
                                        (sp.cos(lam2_val*chi) + sp.I*sp.sin(lam2_val*chi)) * 
                                        Prop.subs(Res, R1_symbol) * Prop.subs(Res, R2_symbol) *
                                        BW.subs(L, LB_val).subs(rBW, rB).subs(qR, sp.Symbol('B')) *
                                        BW.subs(L, L1_val).subs(rBW, r2).subs(qR, R1_symbol) * 
                                        BW.subs(L, J2_val).subs(rBW, r2).subs(qR, R2_symbol)
                                        )
                                AR1R2 += term
                                print(f"\nR1 = {R1_name}, (spin = {R1_spin}), R2 = {R2_name}, (spin = {R2_spin}), lamda = {lam_val}, lam2 = {lam2_val}")
                                print(f"LB = {LB_val}, L1 = {L1}, L2 = {L2}")
                                print(f"term = {term}")

# Mostra l'ampiezza totale
print("\n" + "="*50)
print("Ampiezza AR1R2 totale:")
if AR1R2 == 0:
    print("L'ampiezza AR1R2 è nulla (nessuna risonanza selezionata).")
else:
    print(f"Termini nell'ampiezza AR1R2: {len(AR1R2.as_ordered_terms())}")
print("="*50)



Calcolando contributi per J1 = 0, LB = 1

R1 = K(1460), (spin = 0), R2 = rho(770), (spin = 1), lamda = 0, lam2 = -1
LB = 1, L1 = 1, L2 = 1
term = sqrt(3)*a17*(-I*sin(chi) + cos(chi))*BW(1, r2, K(1460))*BW(1, r2, rho(770))*BW(1, rB, B)*H(eta, 0, 0, 1, K(1460))*LHA(eta, lambda1, lambda2)*R(K(1460))*R(rho(770))*WignerD(0, 0, -1, 0, theta1, 0)*WignerD(1, -1, 0, 0, theta2, 0)*WignerD(1, 0, lambda1 - lambda2, 0, thetal, 0)/(4*pi)

R1 = K(1460), (spin = 0), R2 = rho(770), (spin = 1), lamda = 0, lam2 = 0
LB = 1, L1 = 1, L2 = 1
term = -sqrt(3)*a17*BW(1, r2, K(1460))*BW(1, r2, rho(770))*BW(1, rB, B)*H(eta, 0, 0, 1, K(1460))*LHA(eta, lambda1, lambda2)*R(K(1460))*R(rho(770))*WignerD(0, 0, 0, 0, theta1, 0)*WignerD(1, 0, 0, 0, theta2, 0)*WignerD(1, 0, lambda1 - lambda2, 0, thetal, 0)/(4*pi)

R1 = K(1460), (spin = 0), R2 = rho(770), (spin = 1), lamda = 0, lam2 = 1
LB = 1, L1 = 1, L2 = 1
term = sqrt(3)*a17*(I*sin(chi) + cos(chi))*BW(1, r2, K(1460))*BW(1, r2, rho(770))*BW(1, rB, B)*H(eta, 0, 0, 1, K(

In [7]:
b = sp.symbols('b', real = True)
ml = sp.symbols('m_l', real = True)

Leptonic_subs = {
    ###Leptonic Helicity Amplitudes MASSLESS
    LHA.subs({eta_par: -1, lam1: sp.Rational(-1,2), lam2: sp.Rational(-1,2)}): 0,
    LHA.subs({eta_par: -1, lam1: sp.Rational(-1,2), lam2: sp.Rational(1,2)}): (2)*sp.sqrt(2)*q,
    LHA.subs({eta_par: -1, lam1: sp.Rational(1,2), lam2: sp.Rational(-1,2)}): 0,
    LHA.subs({eta_par: -1, lam1: sp.Rational(1,2), lam2: sp.Rational(1,2)}): 0,

    LHA.subs({eta_par: 1, lam1: sp.Rational(-1,2), lam2: sp.Rational(-1,2)}): 0,
    LHA.subs({eta_par: 1, lam1: sp.Rational(-1,2), lam2: sp.Rational(1,2)}): 0,
    LHA.subs({eta_par: 1, lam1: sp.Rational(1,2), lam2: sp.Rational(-1,2)}): (2)*sp.sqrt(2)*q,    
    LHA.subs({eta_par: 1, lam1: sp.Rational(1,2), lam2: sp.Rational(1,2)}): 0,
}

In [ ]:
# AMPIEZZA TOTALE

print("Verificando la presenza di file salvati...")

# Determina il nome del file
atotsq_file = None

# Cerca i file .pkl nella directory corrente
for file in os.listdir(os.getcwd()):
    if file.startswith("Atotsq_terms") and file.endswith(".pkl"):
        atotsq_file = file

# Controlla se entrambi i file esistono
if atotsq_file and os.path.exists(atotsq_file):
    print("Founded files. Import of saved data.")
    
    try:
        # Carica Atotsq
        with open(atotsq_file, 'rb') as f:
            Atotsqfile = dill.load(f)
            Atotsq = Atotsqfile['Atotsq']
            term_list = Atotsqfile['terms']
            lenght = Atotsqfile['length']
            print("File loaded successfully.")
        winsound.Beep(1000, 2000)
        
    except Exception as e:
        print(f"❌ Errore durante il caricamento: {e}")
        print("Procedo con il calcolo dall'inizio...")
        atotsq_file = None  # Forza il ricalcolo
        
else:
    print("❌ File non trovato. Procedo con il calcolo dell'ampiezza...")

if not atotsq_file:
    print("Calcolando l'ampiezza totale...")
    
    Atot = a_NR + A3 + AR + AR1R2

    totalterms = len(Atot.as_ordered_terms())
    print(f"Numero totale di termini nell'ampiezza Atot: {totalterms}")

    # Sostituisci solo i termini BW con L=0 (non L=1, non L=2)
    W1, W2 = sp.Wild('W1'), sp.Wild('W2')
    Atot = Atot.replace(BWfunc(0, W1, W2), 1)
    
    # Monitora l'uso della memoria
    process = psutil.Process()
    print(f"Memoria utilizzata: {process.memory_info().rss / 1024 / 1024:.1f} MB")
    print(f"Memoria totale: {psutil.virtual_memory().total / 1024 / 1024:.1f} MB")

    # AAcL = (Atot * sp.conjugate(Atot)).xreplace({eta: -1}).xreplace(Leptonic_subs).doit()
    # AAcR = (Atot * sp.conjugate(Atot)).xreplace({eta: 1}).xreplace(Leptonic_subs).doit()

    AAcL = 0
    AAcR = 0
    for lambda1_val in [sp.Rational(-1,2), sp.Rational(1,2)]:
        for lambda2_val in [sp.Rational(-1,2), sp.Rational(1,2)]:
            temp_L = 0
            temp_L = (
                (Atot*sp.conjugate(Atot)).subs({eta: -1, lambda1: lambda1_val, lambda2: lambda2_val}).xreplace(Leptonic_subs).doit()
            )
            print("="*25)
            print(f"Numero termini per eta=-1, lambda1={lambda1_val}, lambda2={lambda2_val}: {len(temp_L.as_ordered_terms())}")
            print(temp_L)
            AAcL += temp_L

            temp_R = 0
            temp_R = (
                (Atot*sp.conjugate(Atot)).subs({eta: 1, lambda1: lambda1_val, lambda2: lambda2_val}).xreplace(Leptonic_subs).doit()
            )
            print("="*25)
            print(f"Numero termini per eta=1, lambda1={lambda1_val}, lambda2={lambda2_val}: {len(temp_R.as_ordered_terms())}")
            print(temp_R)
            AAcR += temp_R

    print("\nEspandendo l'espressione finale.")
    Atotsq = sp.expand(AAcL + AAcR, trig = True)

    # Calcola i termini UNA VOLTA SOLA
    terms_list = Atotsq.as_ordered_terms()
    lenght = len(terms_list)
    print(f"Numero di termini: {lenght}\n")
    print(f"Numero di parametri a_i: {len(atot)}\n")

    for k in range(0,5):
        rnd = randint(0, lenght - 1)
        print(f"Termine casuale #{rnd}:")
        sp.pprint(terms_list[rnd])
        print()

    # Salva ENTRAMBI: l'espressione E i termini già ordinati
    with open(f"Atotsq_terms{lenght}.pkl", 'wb') as f:
        dill.dump({'Atotsq': Atotsq, 'terms': terms_list, 'length': lenght}, f)

    winsound.Beep(1000, 2000)

Verificando la presenza di file salvati...
Founded files. Import of saved data.


In [ ]:
sorted_keys_file = None
trig_coefficients_file = None

# Cerca i file .pkl nella directory corrente
for file in os.listdir(os.getcwd()):
    if file.startswith("sorted_keys") and file.endswith(".pkl"):
        sorted_keys_file = file
    elif file.startswith("trig_coefficients") and file.endswith(".pkl"):
        trig_coefficients_file = file

# Controlla se entrambi i file esistono
if sorted_keys_file and os.path.exists(sorted_keys_file) and trig_coefficients_file and os.path.exists(trig_coefficients_file):
    print("Founded files. Import of saved data.")
    
    try:
        # Carica sorted_keys
        with open(sorted_keys_file, 'rb') as f:
            sorted_keys = dill.load(f)
            print("File loaded successfully.")
        # Carica trig_coefficients
        with open(trig_coefficients_file, 'rb') as f:
            trig_coefficients = dill.load(f)
            print("File loaded successfully.")

        print(f"- Prodotti trigonometrici distinti: {len([k for k, v in trig_coefficients.items() if v != 0])}")
        for k in range(len(sorted_keys)):
            print(f"  • Prodotto #{k+1}: {sorted_keys[k]}")

        winsound.Beep(1000, 500)  # Beep breve per indicare caricamento completato

    except Exception as e:
        print(f"❌ Errore durante il caricamento: {e}")
        print("Procedo con il calcolo dall'inizio...")
        sorted_keys_file = None  # Forza il ricalcolo
        trig_coefficients_file = None  # Forza il ricalcolo

else:
    print("❌ File non trovato. Procedo con l'analisi dei termini trigonometrici...")

if not sorted_keys_file or not trig_coefficients_file:
    def extract_trig_structure(expr):
        """Estrae la struttura trigonometrica completa da un'espressione"""
        trig_factors = []
        non_trig_factors = []
        
        # Scompone il termine in fattori
        if expr.is_Mul: #controlla se è un prodotto 'Mul' per Sympy e li ordina come termini
            factors = expr.as_ordered_factors()
        elif expr.is_Pow:
            # Gestisci le potenze
            base, exp = expr.as_base_exp() # Divide in base ed esponente
            if base.atoms(sp.sin, sp.cos):
                return [expr], []  # Tutta la potenza è trigonometrica
            else:
                return [], [expr]  # Tutta la potenza è non-trigonometrica
        else:
            factors = [expr] #in questo caso ci sono più termini di tipo 'Add' sommati tra loro e li mette in una lista
        
        for factor in factors:
            if factor.atoms(sp.sin, sp.cos):
                # Questo fattore contiene funzioni trigonometriche
                if factor.is_Pow:
                    # È una potenza di funzione trigonometrica
                    trig_factors.append(factor)
                elif factor.is_Mul:
                    # È un prodotto che contiene trigonometria
                    sub_trig, sub_non_trig = extract_trig_structure(factor)
                    trig_factors.extend(sub_trig) #extend aggiunge in coda tutti gli elementi della lista 'sub_trig'
                    non_trig_factors.extend(sub_non_trig)
                else:
                    # È una singola funzione trigonometrica
                    trig_factors.append(factor)
            else:
                non_trig_factors.append(factor)
        
        return trig_factors, non_trig_factors

    # Raccoglie tutti i prodotti trigonometrici unici
    trig_coefficients = {}

    print("Analizzando ogni termine dell'espressione...")
    # Usa terms_list se disponibile, altrimenti calcola
    if 'terms_list' not in locals() or terms_list is None:
        terms_list = Atotsq.as_ordered_terms()
        lenght = len(terms_list)
    
    for i, term in tqdm(enumerate(terms_list), total=lenght, desc="Analisi termini angolari"):
        
        # Estrai la struttura trigonometrica
        trig_factors, non_trig_factors = extract_trig_structure(term)
        
        # Costruisci il prodotto trigonometrico
        if trig_factors:
            trig_product = 1
            for tf in trig_factors:
                trig_product *= tf
            trig_key = str(trig_product)
        else:
            trig_key = "COSTANTE"
        
        # Costruisci il coefficiente non trigonometrico
        if non_trig_factors:
            non_trig_product = 1
            for ntf in non_trig_factors:
                non_trig_product *= ntf
        else:
            non_trig_product = 1
        
        # Accumula i coefficienti
        if trig_key in trig_coefficients:
            trig_coefficients[trig_key] += non_trig_product
        else:
            trig_coefficients[trig_key] = non_trig_product

    print(f"\nTrovati {len(trig_coefficients)} prodotti trigonometrici distinti")

    # print("Semplificando i coefficienti...")
    # for key in tqdm(trig_coefficients, desc="Semplificazione coefficienti", total=len(trig_coefficients)):
    #     trig_coefficients[key] = sp.simplify(trig_coefficients[key])


    # Ordina per lunghezza della stringa (prima i più semplici)
    print("Ordinando i prodotti trigonometrici per complessità")
    sorted_keys = sorted(trig_coefficients.keys(), key=lambda x: (len(x), x))

    print(f"\n" + "="*70)
    print("RIEPILOGO FINALE:")
    print("="*70)
    print(f"- Prodotti trigonometrici distinti: {len([k for k, v in trig_coefficients.items() if v != 0])}")
    for k in range(len(sorted_keys)):
        print(f"  • Prodotto #{k+1}: {sorted_keys[k]}")

    print(f"- Analisi completata con successo!")

    with open("trig_coefficients.pkl", 'wb') as f:
        dill.dump(trig_coefficients, f)
        print("Dati salvati in 'trig_coefficients.pkl'")

    with open("sorted_keys.pkl", 'wb') as f:
        dill.dump(sorted_keys, f)
        print("Chiavi salvate in 'sorted_keys.pkl'")

    winsound.Beep(1500, 2000)

Founded files. Import of saved data.
File loaded successfully.
File loaded successfully.
- Prodotti trigonometrici distinti: 1087
  • Prodotto #1: COSTANTE
  • Prodotto #2: sin(thetal)
  • Prodotto #3: sin(thetal)**2
  • Prodotto #4: sin(phi)*sin(theta1)
  • Prodotto #5: sin(theta1)*cos(phi)
  • Prodotto #6: sin(thetal)*cos(theta1)
  • Prodotto #7: sin(thetal)*cos(theta2)
  • Prodotto #8: sin(phi)**2*sin(theta1)**2
  • Prodotto #9: sin(theta1)**2*cos(phi)**2
  • Prodotto #10: sin(thetal)**2*cos(theta1)
  • Prodotto #11: sin(thetal)**2*cos(theta2)
  • Prodotto #12: sin(thetal)*cos(theta1)**2
  • Prodotto #13: sin(thetal)*cos(theta2)**2
  • Prodotto #14: sin(chi)*sin(phi)*sin(theta2)
  • Prodotto #15: sin(chi)*sin(theta2)*cos(phi)
  • Prodotto #16: sin(phi)*sin(theta2)*cos(chi)
  • Prodotto #17: sin(theta2)*cos(chi)*cos(phi)
  • Prodotto #18: sin(thetal)**2*cos(theta1)**2
  • Prodotto #19: sin(thetal)**2*cos(theta1)**3
  • Prodotto #20: sin(thetal)**2*cos(theta1)**4
  • Prodotto #21: sin

In [ ]:
chirality_file = None

# Cerca file chirality*.pkl
for file in os.listdir(os.getcwd()):
    if file.startswith("chirality") and file.endswith(".pkl"):
        chirality_file = file

if chirality_file and os.path.exists(chirality_file):
    print("Founded files. Import of saved data.")
    try:
        with open(chirality_file, 'rb') as f:
            chirality = dill.load(f)
            print("File loaded successfully.")

            print(f"Chirality of terms:")
            for k in range(len(sorted_keys)):
                print(f"f_{k+1} = {sorted_keys[k]}. Chiralità = {chirality.get(sorted_keys[k])}")


            winsound.Beep(1000, 500)
    except Exception as e:
        print(f"❌ Errore durante il caricamento: {e}")
        print("Procedo con il calcolo dall'inizio...")
        chirality_file = None

else:
    chirality = {}

    for idx_key, key in tqdm(
        enumerate(sorted_keys),
        desc="Trovando la chiralità",
        total=len(sorted_keys)
    ):

        if key == "COSTANTE":
            chirality[key] = 1
            print(f"Key #{idx_key+1}: {key} --> Chirality = 1 (COSTANTE)")
            continue

        # ... dentro il ciclo principale for key in ... ...
        
        listed_coeff = trig_coefficients[key].as_ordered_terms()
        first_term = listed_coeff[0]
        # Crea wildcard per le risonanze
        w_res = sp.Wild('w_res')
        
        excluded_names = {'H', 'G', 'Hbar'}

        # --- 1. Calcolo il Pure Term di Riferimento ---
        ref_pure = 1
        for factor in first_term.as_ordered_factors():
            is_excluded = False
            atoms_in_factor = factor.atoms(sp.Function, sp.Symbol)
            for atom in atoms_in_factor:
                name = atom.name if hasattr(atom, 'name') else atom.func.__name__
                if name in excluded_names:
                    is_excluded = True
                    break
            if not is_excluded:
                ref_pure *= factor

        # --- 2. Cerco il partner per determinare la Chiralità ---
        # Default: se non trovo nulla, decidi tu cosa mettere (es. None o 0)
        detected_chirality = None 
        
        # Itero sugli altri termini (salto il primo che è il riferimento)
        for term in listed_coeff[1:]:
            current_pure = 1
            for factor in term.as_ordered_factors():
                is_excluded = False
                atoms_in_factor = factor.atoms(sp.Function, sp.Symbol)
                for atom in atoms_in_factor:
                    name = atom.name if hasattr(atom, 'name') else atom.func.__name__
                    if name in excluded_names:
                        is_excluded = True
                        break
                if not is_excluded:
                    current_pure *= factor
            
            # --- CONFRONTO DEI SEGNI ---
            
            # Caso A: Segno OPPOSTO (Antisimmetrico) -> Es. Chiralità -1
            if current_pure == -ref_pure:
                detected_chirality = -1 
                break # Trovato, possiamo uscire
            
            # Caso B: Segno UGUALE (Simmetrico) -> Es. Chiralità +1
            elif current_pure == ref_pure:
                detected_chirality = 1
                break

        # Se non ha trovato partner (es. termine unico), gestisci il caso:
        if detected_chirality is None:
            print("  -> Nessun termine partner trovato (termine singolo).")
            detected_chirality = 0 # O quello che ha senso fisico per te

        chirality[key] = detected_chirality
        print(f"Key #{idx_key+1}: {key} --> Chirality = {detected_chirality}")




    # Salvataggio finale
    with open("chirality.pkl", 'wb') as f:
        dill.dump(chirality, f)

    winsound.Beep(1500, 1000)

Founded files. Import of saved data.
File loaded successfully.
Chirality of terms:
f_1 = COSTANTE. Chiralità = 1
f_2 = sin(thetal). Chiralità = -1
f_3 = sin(thetal)**2. Chiralità = 1
f_4 = sin(phi)*sin(theta1). Chiralità = 1
f_5 = sin(theta1)*cos(phi). Chiralità = -1
f_6 = sin(thetal)*cos(theta1). Chiralità = -1
f_7 = sin(thetal)*cos(theta2). Chiralità = -1
f_8 = sin(phi)**2*sin(theta1)**2. Chiralità = 1
f_9 = sin(theta1)**2*cos(phi)**2. Chiralità = -1
f_10 = sin(thetal)**2*cos(theta1). Chiralità = 1
f_11 = sin(thetal)**2*cos(theta2). Chiralità = 1
f_12 = sin(thetal)*cos(theta1)**2. Chiralità = -1
f_13 = sin(thetal)*cos(theta2)**2. Chiralità = -1
f_14 = sin(chi)*sin(phi)*sin(theta2). Chiralità = -1
f_15 = sin(chi)*sin(theta2)*cos(phi). Chiralità = 1
f_16 = sin(phi)*sin(theta2)*cos(chi). Chiralità = 1
f_17 = sin(theta2)*cos(chi)*cos(phi). Chiralità = -1
f_18 = sin(thetal)**2*cos(theta1)**2. Chiralità = 1
f_19 = sin(thetal)**2*cos(theta1)**3. Chiralità = 1
f_20 = sin(thetal)**2*cos(theta

In [ ]:
LEFT_trig_coefficients_file = None

# Cerca file LEFT_trig_coefficients*.pkl
for file in os.listdir(os.getcwd()):
    if file.startswith("LEFT_trig_coefficients") and file.endswith(".pkl"):
        LEFT_trig_coefficients_file = file

if LEFT_trig_coefficients_file and os.path.exists(LEFT_trig_coefficients_file):
    print("Founded files. Import of saved data.")
    try:
        with open(LEFT_trig_coefficients_file, 'rb') as f:  # FIX: era chirality_file
            LEFT_trig_coefficients = dill.load(f)
            print("File LEFT_trig_coefficients caricato correttamente.")
            winsound.Beep(1000, 500)
    except Exception as e:
        print(f"❌ Errore durante il caricamento: {e}")
        print("Procedo con il calcolo dall'inizio...")
        LEFT_trig_coefficients_file = None

if not LEFT_trig_coefficients_file:
    # Filtra i termini con eta=-1 da tutti i coefficienti
    LEFT_trig_coefficients = {}

    print("Filtrando i termini con eta=-1 (Left-handed)...")

    for key in tqdm(sorted_keys, desc="Filtraggio termini L", total=len(sorted_keys)):
        if key == "COSTANTE":
            LEFT_trig_coefficients[key] = sp.Rational(1, 2) * trig_coefficients[key]
            continue
        
        coeff = trig_coefficients[key]
        
        if coeff == 0:
            LEFT_trig_coefficients[key] = 0
            continue
        
        terms = coeff.as_ordered_terms()
        filtered_terms = []
        
        for term in terms:
            has_L = False
            
            for factor in term.as_ordered_factors():
                # Funzione helper per estrarre eta da H, Hbar, G
                def check_eta(func_obj):
                    if hasattr(func_obj, 'func'):
                        func_name = func_obj.func.__name__ if hasattr(func_obj.func, '__name__') else str(func_obj.func)
                        if func_name in ['H', 'Hbar', 'G']:
                            if len(func_obj.args) > 0:
                                return func_obj.args[0]  # Ritorna il valore di eta
                    return None
                
                # Caso 1: Funzione diretta H(-1, ...), Hbar(-1, ...), G(-1, ...)
                eta_val = check_eta(factor)
                if eta_val == -1:
                    has_L = True
                    break
                
                # Caso 2: Coniugato conjugate(H(-1, ...))
                if hasattr(factor, 'func') and factor.func.__name__ == 'conjugate':
                    # Estrai la funzione interna dal coniugato
                    inner_func = factor.args[0]  # Questo è H(-1, ...) dentro conjugate(...)
                    eta_val = check_eta(inner_func)
                    if eta_val == -1:
                        has_L = True
                        break
            
            if has_L:
                filtered_terms.append(term)
        
        if filtered_terms:
            LEFT_trig_coefficients[key] = sum(filtered_terms)
        else:
            LEFT_trig_coefficients[key] = 0

    # Statistiche
    print(f"\n" + "="*70)
    print("RISULTATI FILTRAGGIO:")
    print("="*70)

    original = 0
    left = 0
    for k in range(len(sorted_keys)):
        original += len(trig_coefficients[sorted_keys[k]].as_ordered_terms())
        left += len(LEFT_trig_coefficients[sorted_keys[k]].as_ordered_terms())
    print(f"- Termini totali originali: {original}")
    print(f"- Termini totali Left-handed: {left}")

    # Salva
    with open("LEFT_trig_coefficients.pkl", 'wb') as f:
        dill.dump(LEFT_trig_coefficients, f)
        print(f"\nDati salvati in 'LEFT_trig_coefficients.pkl'")

    winsound.Beep(1000, 500)

Filtrando i termini con eta=-1 (Left-handed)...


Filtraggio termini L:   0%|          | 0/1087 [00:00<?, ?it/s]


RISULTATI FILTRAGGIO:
- Termini totali originali: 736391
- Termini totali Left-handed: 368196

Dati salvati in 'LEFT_trig_coefficients.pkl'


In [ ]:
print("Semplificando i coefficienti: z × z* → |z|²...")

SIMPLIFIED_LEFT_trig_coefficients_file = None

# Cerca file SIMPLIFIED_LEFT_trig_coefficients*.pkl
for file in os.listdir(os.getcwd()):
    if file.startswith("SIMPLIFIED_LEFT_trig_coefficients") and file.endswith(".pkl"):
        SIMPLIFIED_LEFT_trig_coefficients_file = file

if SIMPLIFIED_LEFT_trig_coefficients_file and os.path.exists(SIMPLIFIED_LEFT_trig_coefficients_file):
    print("Founded files. Import of saved data.")
    try:
        with open(SIMPLIFIED_LEFT_trig_coefficients_file, 'rb') as f:
            SIMPLIFIED_LEFT_trig_coefficients = dill.load(f)
            print("File SIMPLIFIED_LEFT_trig_coefficients caricato correttamente.")
            winsound.Beep(1000, 500)
    except Exception as e:
        print(f"❌ Errore durante il caricamento: {e}")
        print("Procedo con il calcolo dall'inizio...")
        SIMPLIFIED_LEFT_trig_coefficients_file = None

if not SIMPLIFIED_LEFT_trig_coefficients_file:
    SIMPLIFIED_LEFT_trig_coefficients = {}

    for key in tqdm(sorted_keys, desc="Semplificazione algebrica", total=len(sorted_keys)):
        
        coeff = LEFT_trig_coefficients[key]
        
        if coeff == 0:
            SIMPLIFIED_LEFT_trig_coefficients[key] = 0
            continue
        
        # Scomponi in termini
        terms = coeff.as_ordered_terms()
        simplified_terms = []
        
        for term in terms:
            # Scomponi il termine in fattori
            factors = list(term.as_ordered_factors())
            
            # Cerca coppie z e conjugate(z)
            new_factors = []
            used_indices = set()
            
            for i, factor_i in enumerate(factors):
                if i in used_indices:
                    continue
                
                # Cerca se esiste il coniugato di questo fattore
                found_conjugate = False
                
                for j, factor_j in enumerate(factors):
                    if i == j or j in used_indices:
                        continue
                    
                    # Caso 1: factor_j è conjugate(factor_i)
                    if (hasattr(factor_j, 'func') and 
                        factor_j.func.__name__ == 'conjugate' and 
                        len(factor_j.args) > 0 and 
                        factor_j.args[0] == factor_i):
                        
                        # Sostituisci con |factor_i|²
                        new_factors.append(sp.Abs(factor_i)**2)
                        used_indices.add(i)
                        used_indices.add(j)
                        found_conjugate = True
                        break
                    
                    # Caso 2: factor_i è conjugate(factor_j)
                    if (hasattr(factor_i, 'func') and 
                        factor_i.func.__name__ == 'conjugate' and 
                        len(factor_i.args) > 0 and 
                        factor_i.args[0] == factor_j):
                        
                        # Sostituisci con |factor_j|²
                        new_factors.append(sp.Abs(factor_j)**2)
                        used_indices.add(i)
                        used_indices.add(j)
                        found_conjugate = True
                        break
                
                if not found_conjugate:
                    new_factors.append(factor_i)
                    used_indices.add(i)
            
            # Ricostruisci il termine
            if new_factors:
                simplified_term = sp.Mul(*new_factors)
                simplified_terms.append(simplified_term)
        
        # Ricostruisci il coefficiente
        if simplified_terms:
            SIMPLIFIED_LEFT_trig_coefficients[key] = sum(simplified_terms)
            
        else:
            SIMPLIFIED_LEFT_trig_coefficients[key] = 0

    # Salva
    with open("SIMPLIFIED_LEFT_trig_coefficients.pkl", 'wb') as f:
        dill.dump(SIMPLIFIED_LEFT_trig_coefficients, f)
        print(f"\n{'='*70}")
        print(f"Dati salvati in 'SIMPLIFIED_LEFT_trig_coefficients.pkl'")

    winsound.Beep(1000, 500)

Semplificando i coefficienti: z × z* → |z|²...


Semplificazione algebrica:   0%|          | 0/1087 [00:00<?, ?it/s]


Dati salvati in 'SIMPLIFIED_LEFT_trig_coefficients.pkl'


In [ ]:
print("Semplificando termini coniugati: z±conj(z) → 2Re(z) o 2iIm(z)...")

FINAL_trig_coefficients_file = None
for file in os.listdir(os.getcwd()):
    if file.startswith("FINAL_trig_coefficients") and file.endswith(".pkl"):
        FINAL_trig_coefficients_file = file

if FINAL_trig_coefficients_file and os.path.exists(FINAL_trig_coefficients_file):
    print("Founded files. Import of saved data.")
    try:
        with open(FINAL_trig_coefficients_file, 'rb') as f:
            FINAL_trig_coefficients = dill.load(f)
            print("File FINAL_trig_coefficients caricato correttamente.")
            winsound.Beep(1000, 500)
    except Exception as e:
        print(f"❌ Errore durante il caricamento: {e}")
        print("Procedo con il calcolo dall'inizio...")
        FINAL_trig_coefficients_file = None

if not FINAL_trig_coefficients_file:

    FINAL_trig_coefficients = {}

    for key in tqdm(sorted_keys, desc="Semplificazione z±conj(z)", total=len(sorted_keys)):
        
        coeff = SIMPLIFIED_LEFT_trig_coefficients[key]
        
        if coeff == 0:
            FINAL_trig_coefficients[key] = 0
            continue
        
        # Scomponi in termini
        terms = list(coeff.as_ordered_terms())
        
        new_terms = []
        used_indices = set()
        
        for i, term_i in enumerate(terms):
            if i in used_indices:
                continue
            
            found_conjugate = False
            
            # Cerca il coniugato tra gli altri termini
            for j, term_j in enumerate(terms):
                if i == j or j in used_indices:
                    continue
                
                # Controlla se term_j == conjugate(term_i) -> SOMMA
                if term_j == sp.conjugate(term_i):
                    used_indices.add(i)
                    used_indices.add(j)
                    found_conjugate = True
                    # z + conj(z) = 2*Re(z)
                    new_terms.append(2 * sp.re(term_i))
                    break
                
                # Controlla se term_j == -conjugate(term_i) -> DIFFERENZA
                elif term_j == -sp.conjugate(term_i):
                    used_indices.add(i)
                    used_indices.add(j)
                    found_conjugate = True
                    # z - conj(z) = 2i*Im(z), ma siccome z contiene già I,
                    # scriviamo direttamente: (z - conj(z))/i = 2*Im(z) reale
                    # Oppure lasciamo: i*(z - conj(z))/2i = (z - conj(z))/2
                    # Meglio: z - conj(z) è puramente immaginario, quindi:
                    new_terms.append(term_i - sp.conjugate(term_i))
                    break
            
            if not found_conjugate:
                # Nessun coniugato trovato, mantieni il termine
                new_terms.append(term_i)
                used_indices.add(i)
        
        # Ricostruisci il coefficiente
        if new_terms:
            FINAL_trig_coefficients[key] = sum(new_terms)

        else:
            FINAL_trig_coefficients[key] = 0

    # Salva
    with open("FINAL_trig_coefficients.pkl", 'wb') as f:
        dill.dump(FINAL_trig_coefficients, f)
        print(f"\nDati salvati in 'FINAL_trig_coefficients.pkl'")

    winsound.Beep(1500, 500)

Semplificando termini coniugati: z±conj(z) → 2Re(z) o 2iIm(z)...


Semplificazione z±conj(z):   0%|          | 0/1087 [00:00<?, ?it/s]


Dati salvati in 'FINAL_trig_coefficients.pkl'


In [25]:
piu = 0
meno = 0
print("Primi cinque termini con chirality 1:")
for index, key in enumerate(sorted_keys):
    if chirality[key] == 1 and piu < 6:
        print(f"\n\n{index+1}:{key}:")
        sp.pprint(FINAL_trig_coefficients[key])
        piu += 1

print("Primi cinque termini con chirality -1:")
for index, key in enumerate(sorted_keys):
    if chirality[key] == -1 and meno < 6:
        print(f"\n\n{index+1}:{key}:")
        sp.pprint(FINAL_trig_coefficients[key])
        meno += 1

Primi cinque termini con chirality 1:


1:COSTANTE:
        2
4⋅│a_NR│ 


3:sin(thetal)**2:
 2   2             2                     ⎛                                         ___ ________________________ ______ ↪
q ⋅BW (1, rB, B)⋅BW (2, r₂, K2*(1430))⋅re⎝a₂₉⋅H(-1, 2, 0, 1, K2(1700))⋅R(K2(1700))⋅a₃₁⋅H(-1, 2, 0, 1, K2(1820))⋅R(K2(1 ↪
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────── ↪
                                                                       2                                               ↪
                                                                   16⋅π                                                ↪

↪ _____⎞               2    2   2             2                        2                           2              2    ↪
↪ 820))⎠⋅│R(K2*(1430))│    q ⋅BW (1, rB, B)⋅BW (2, r₂, K2*(1430))⋅│a₂₉│ ⋅│H(-1, 2, 0, 1, K2(1700))│ ⋅│R(K2(1700))│ ⋅│R ↪
↪ ────────────────────── + ─────────────────────────────────

In [ ]:
### ANALISI DELLE CHIAVI TRIGONOMETRICHE PER RIDONDANZE ###
print("Verificando se le chiavi trigonometriche sono tutte realmente distinte...")
print("="*70)

def analyze_trig_keys_for_redundancy(sorted_keys, trig_coefficients):
    """
    Analizza le chiavi trigonometriche per identificare possibili ridondanze
    """
    # Filtra solo chiavi con coefficienti non nulli
    non_zero_keys = [k for k in sorted_keys if trig_coefficients[k] != 0]
    
    print(f"Chiavi totali: {len(sorted_keys)}")
    print(f"Chiavi con coefficiente non nullo: {len(non_zero_keys)}")
    
    # 1. Categorizza le chiavi
    print("\n" + "="*40)
    print("CATEGORIZZAZIONE:")
    print("="*40)
    
    categories = {
        'costanti': [],
        'semplici': [],  # sin(x), cos(x)
        'potenze': [],   # sin²(x), cos²(x) 
        'prodotti_2': [],  # sin(x)*cos(y)
        'prodotti_complessi': []  # più di 2 fattori
    }
    
    for key in non_zero_keys:
        if key == "COSTANTE":
            categories['costanti'].append(key)
        elif '*' not in key and '**' not in key:
            categories['semplici'].append(key)
        elif '**' in key and key.count('*') <= 2:  # es: sin(x)**2
            categories['potenze'].append(key)
        elif key.count('*') == 1:  # esattamente un prodotto
            categories['prodotti_2'].append(key)
        else:
            categories['prodotti_complessi'].append(key)
    
    for cat_name, cat_keys in categories.items():
        print(f"{cat_name}: {len(cat_keys)}")
        if len(cat_keys) <= 5:
            for k in cat_keys:
                print(f"  • {k}")
        else:
            for k in cat_keys[:3]:
                print(f"  • {k}")
            print(f"  ... e altri {len(cat_keys)-3}")
    
    # 2. Controlli per identità trigonometriche
    print("\n" + "="*40)
    print("CONTROLLO IDENTITÀ TRIGONOMETRICHE:")
    print("="*40)
    
    # Cerca coppie cos²(x) + sin²(x) = 1
    cos_squared = [k for k in categories['potenze'] if 'cos(' in k and '**2' in k]
    sin_squared = [k for k in categories['potenze'] if 'sin(' in k and '**2' in k]
    
    print(f"\nTermini cos²(x): {len(cos_squared)}")
    print(f"Termini sin²(x): {len(sin_squared)}")
    
    complementary_pairs = []
    for cos_term in cos_squared:
        # Estrai argomento: cos(theta1)**2 -> theta1
        arg_start = cos_term.find('(') + 1
        arg_end = cos_term.find(')')
        if arg_start > 0 and arg_end > arg_start:
            arg = cos_term[arg_start:arg_end]
            expected_sin = f"sin({arg})**2"
            if expected_sin in sin_squared:
                complementary_pairs.append((cos_term, expected_sin))
    
    if complementary_pairs:
        print(f"\n COPPIE COMPLEMENTARI TROVATE ({len(complementary_pairs)}):")
        for cos_t, sin_t in complementary_pairs:
            coeff_cos = trig_coefficients.get(cos_t, 0)
            coeff_sin = trig_coefficients.get(sin_t, 0)
            total_coeff = coeff_cos + coeff_sin
            print(f"  • {cos_t} + {sin_t}")
            print(f"    Coefficienti: {coeff_cos} + {coeff_sin} = {sp.simplify(total_coeff)}")
    else:
        print("✅ Nessuna coppia complementaria cos²+sin²=1 trovata")
    
    # 3. Controllo prodotti con stessi argomenti
    print(f"\n3. Controllo prodotti con variabili comuni:")
    
    # Raggruppa per variabili angolari presenti
    angle_groups = {}
    import re
    
    for key in non_zero_keys:
        if key != "COSTANTE":
            # Trova tutte le variabili angolari nel termine
            angles = set(re.findall(r'(theta1|theta2|thetal|phi|chi)', key))
            if angles:
                angle_signature = tuple(sorted(angles))
                if angle_signature not in angle_groups:
                    angle_groups[angle_signature] = []
                angle_groups[angle_signature].append(key)
    
    print(f"Gruppi di termini con variabili comuni: {len(angle_groups)}")
    
    for signature, terms in angle_groups.items():
        if len(terms) > 1:
            print(f"\nVariabili {signature}: {len(terms)} termini")
            for term in terms[:3]:  # Mostra solo i primi 3
                print(f"  • {term}")
            if len(terms) > 3:
                print(f"  ... e altri {len(terms)-3}")
    
    # 4. Test di semplificazione SymPy
    print(f"\n4. Test di semplificazione automatica SymPy:")
    
    # Prova a convertire alcune chiavi in espressioni SymPy e semplificarle
    test_keys = non_zero_keys[:10]  # Testa solo le prime 10
    
    for key in test_keys:
        if key != "COSTANTE":
            try:
                expr = sp.sympify(key)
                simplified = sp.simplify(expr)
                trigsimp = sp.trigsimp(expr)
                
                if str(simplified) != key or str(trigsimp) != key:
                    print(f"  • {key}")
                    if str(simplified) != key:
                        print(f"    simplify() → {simplified}")
                    if str(trigsimp) != key:
                        print(f"    trigsimp() → {trigsimp}")
            except:
                continue
    
    return categories, complementary_pairs, angle_groups

# Esegui l'analisi se esistono i dati
if 'sorted_keys' in globals() and 'trig_coefficients' in globals():
    categories, pairs, groups = analyze_trig_keys_for_redundancy(sorted_keys, trig_coefficients)
    
    print(f"\n" + "="*70)
    print("CONCLUSIONI:")
    print("="*70)
    if pairs:
        print(f"⚠️  Trovate {len(pairs)} possibili ridondanze cos²+sin²")
    else:
        print("✅ Nessuna ridondanza evidente trovata")
    
    print(f"📊 In totale: {len([k for k, v in trig_coefficients.items() if v != 0])} chiavi trigonometriche distinte")
    
else:
    print("❌ Dati non disponibili. Eseguire prima l'analisi trigonometrica nella cella precedente.")

Verificando se le chiavi trigonometriche sono tutte realmente distinte...
Chiavi totali: 491
Chiavi con coefficiente non nullo: 491

CATEGORIZZAZIONE:
costanti: 1
  • COSTANTE
semplici: 0
potenze: 0
prodotti_2: 2
  • sin(phi)*sin(theta1)
  • sin(theta1)*cos(phi)
prodotti_complessi: 488
  • sin(phi)**2*sin(theta1)**2
  • sin(theta1)**2*cos(phi)**2
  • sin(chi)*sin(phi)*sin(theta2)
  ... e altri 485

CONTROLLO IDENTITÀ TRIGONOMETRICHE:

Termini cos²(x): 0
Termini sin²(x): 0
✅ Nessuna coppia complementaria cos²+sin²=1 trovata

3. Controllo prodotti con variabili comuni:
Gruppi di termini con variabili comuni: 8

Variabili ('phi', 'theta1'): 10 termini
  • sin(phi)*sin(theta1)
  • sin(theta1)*cos(phi)
  • sin(phi)**2*sin(theta1)**2
  ... e altri 7

Variabili ('chi', 'phi', 'theta2'): 24 termini
  • sin(chi)*sin(phi)*sin(theta2)
  • sin(chi)*sin(theta2)*cos(phi)
  • sin(phi)*sin(theta2)*cos(chi)
  ... e altri 21

Variabili ('phi', 'theta1', 'theta2'): 16 termini
  • sin(phi)*sin(theta1)*cos